# Modèle de langage et génération de séquence

Dans cette démo, nous allons étudier les différentes manières de décoder du texte à partir d'un modèle entraîné. Cette étape est nécessaire pour beaucoup de tâches de traitement du langage : traduction, question/réponse, génération…

Nous prendrons l'exemple de la génération de texte. Longtemps, elle a reposé sur des textes à trous plus ou moins sophistiqués. GPT-2 (2019) a montré qu'un réseau de neurones entraîné sur de grandes quantités de texte écrit des paragraphes fluides. C'est un transformer de type *décodeur seul* : les grands modèles de langage actuels (GPT-4, Llama, Mistral…) reposent sur le même principe et produisent leurs réponses avec les méthodes de décodage présentées ici.

Nous partirons de la méthode la plus simple pour progresser jusqu'aux méthodes utilisées en pratique, en codant nous-mêmes le choix du token suivant. Cela montre le rôle de la softmax en sortie des modèles de langage. Pour finir, nous retrouverons tous ces réglages dans la méthode `generate` de KerasHub.

Un [excellent article](https://huggingface.co/blog/how-to-generate) de Hugging Face peut servir de ressource complémentaire à ce notebook.

Cette démo n'a pas besoin de carte graphique : GPT-2 small tourne sur CPU, et chaque test sur les 7 prompts y prend une minute environ. Elle utilise le GPU s'il y en a un.

**Attention** : ces modèles ont été entraînés sur des textes du web, sans filtrage. Les textes générés, surtout par échantillonnage, peuvent être incohérents, faux ou choquants.

## Installation

La bibliothèque [KerasHub](https://keras.io/keras_hub/) met à disposition des modèles pré-entraînés pour Keras, dont GPT-2. Elle est préinstallée sur Colab ; la cellule suivante l'installe ailleurs (Codespace, poste personnel).

In [1]:
!pip install -q keras-hub

## Choix de la langue

Nous allons voir comment générer du texte en anglais mais aussi en français, en chargeant des modèles pré-entraînés sur des corpus différents. La cellule suivante permet de changer la langue utilisée pour le reste de la démo.

In [2]:
langue = "fr"
# lang = "en"

## Import d'un modèle entraîné

Un modèle de KerasHub se charge avec la méthode `from_preset` de sa classe, `keras_hub.models.GPT2CausalLM` pour un GPT-2 qui prédit le token suivant (*causal language model*) : elle récupère l'architecture, les poids et le tokeniseur.

- En anglais, `"gpt2_base_en"` est le GPT-2 small d'OpenAI (2019), publié par KerasHub sur [Kaggle](https://www.kaggle.com/models/keras/gpt2).
- En français, [BelGPT-2](https://huggingface.co/antoinelouis/belgpt2) est un GPT-2 small entraîné sur un corpus français. KerasHub le charge depuis le Hub Hugging Face avec le préfixe `hf://`. Son vocabulaire nomme `</s>` le symbole de fin de texte que KerasHub attend sous le nom `<|endoftext|>` : la cellule construit donc le tokeniseur elle-même, après avoir renommé ce symbole.

Le tokeniseur du modèle se retrouve ensuite dans `causal_lm.preprocessor.tokenizer`.

In [3]:
import functools
import json
import statistics
import warnings

import keras
import keras_hub
import tensorflow as tf
import tqdm.auto
from huggingface_hub import hf_hub_download

# Mêmes tirages aléatoires à chaque exécution du notebook
keras.utils.set_random_seed(0)

if langue == "fr":
  repo = "antoinelouis/belgpt2"
  with open(hf_hub_download(repo, "vocab.json"), encoding="utf-8") as fichier:
    vocabulaire = json.load(fichier)
  # Le symbole de fin de texte, sous le nom que KerasHub attend
  vocabulaire["<|endoftext|>"] = vocabulaire.pop("</s>")
  tokenizer = keras_hub.models.GPT2Tokenizer(
    vocabulaire, hf_hub_download(repo, "merges.txt")
  )
  causal_lm = keras_hub.models.GPT2CausalLM.from_preset(
    f"hf://{repo}",
    preprocessor=keras_hub.models.GPT2CausalLMPreprocessor(tokenizer),
  )
else:
  causal_lm = keras_hub.models.GPT2CausalLM.from_preset("gpt2_base_en")

tokenizer = causal_lm.preprocessor.tokenizer

## Inspection du modèle

Après la récupération d'un modèle pré-entraîné, il est toujours bon de vérifier son architecture et ses caractéristiques. La cellule suivante affiche le résumé du modèle, celui de son *backbone* (le transformer, sans la couche de sortie), la configuration du backbone et celle de son premier bloc. La suivante en extrait les chiffres utiles.

In [4]:
causal_lm.summary()
causal_lm.backbone.summary()
print(causal_lm.backbone.get_config())
print(causal_lm.backbone.get_layer("transformer_layer_0").get_config())

Preprocessor: "gpt2_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gpt2_tokenizer (GPT2Tokenizer)                                │                       Vocab size: 50,257 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gpt2_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gpt2_backbone (GPT2Backbone)  │ (None, None, 768)         │     124,439,808 │ padding_mask[0][0],        │
│                               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 50257)       │      38,597,376 │ gpt2_backbone[0][0]        │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 124,439,808 (474.70 MB)

 Trainable params: 124,439,808 (474.70 MB)

... (3 outputs left out)

 Total params: 124,439,808 (474.70 MB)

 Trainable params: 124,439,808 (474.70 MB)

 Non-trainable params: 0 (0.00 B)

{'name': 'gpt2_backbone', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'vocabulary_size': 50257, 'num_layers': 12, 'num_heads': 12, 'hidden_dim': 768, 'intermediate_dim': 3072, 'dropout': 0.1, 'max_sequence_length': 1024}
{'name': 'transformer_layer_0', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'intermediate_dim': 3072, 'num_heads': 12, 'dropout': 0.1, 'activation': {'module': 'builtins', 'class_name': 'function', 'config': 'keras_hub>gelu_approximate', 'registered_name': 'function'}, 'layer_norm_epsilon': 1e-05, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'RandomNormal', 'config': {'seed': None, 'mean': 0.0, 'stddev': 0.02}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'normalize_first':

In [5]:
config = causal_lm.backbone.get_config()
print(f"Couches de transformer : {config['num_layers']}")
print(f"Dropout : {config['dropout']}")
print(f"{causal_lm.count_params() / 1e6:.0f} millions de paramètres")
print(f"Début/fin de texte : {tokenizer.start_token_id} / {tokenizer.end_token_id}")

Couches de transformer : 12
Dropout : 0.1
124 millions de paramètres
Début/fin de texte : 2 / 2


- Le modèle possède 12 couches de transformer (`num_layers`, ou les blocs `transformer_layer_0` à `transformer_layer_11` du résumé).
- Son mécanisme de régularisation est le dropout, de taux 0,1 (`dropout`, la couche `embeddings_dropout` et le dropout de chaque bloc).
- Ses mécanismes de stabilisation de l'apprentissage sont la normalisation de couche (la couche `layer_norm` en sortie, et dans chaque bloc `layer_norm_epsilon` et `normalize_first` : une normalisation avant l'attention et avant la couche dense) et les connexions résiduelles autour de l'attention et de la couche dense de chaque bloc (internes à `TransformerDecoder`, elles n'apparaissent pas dans le résumé).
- Il a environ 124 millions de paramètres. La couche de sortie `token_embedding (ReversibleEmbedding)` n'en ajoute aucun : elle réutilise la matrice d'embedding des tokens de l'entrée.
- `tokenizer.start_token_id` et `tokenizer.end_token_id` donnent l'index du symbole de début et de fin de texte : 50256 (`<|endoftext|>`) pour GPT-2, 2 pour BelGPT-2. Nous en aurons besoin pour encoder les textes.

## Le mécanisme de génération

Pour générer du texte depuis un modèle entraîné, on procède de manière itérative : on génère les tokens un par un, en commençant par un texte vide ou par une phrase que l'on souhaite compléter.

Pour générer un token, on donne en entrée du réseau les tokens générés jusque-là (sous forme d'indices dans le vocabulaire). Le réseau produit alors un score par token du vocabulaire. On choisit l'un de ces tokens, on l'ajoute à ce qui a déjà été généré, puis on recommence pour générer le token suivant.

Plus formellement, on essaye de maximiser $P(\text{Texte}| \text{Initial})$ avec la décomposition suivante :

$$P(\text{Texte}| \text{Initial}) = \prod_{i=1}^{\text{Taille du texte}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

Tout l'enjeu du décodage est de trouver le texte qui maximise ce produit : c'est la meilleure proposition du modèle. Cette valeur est impossible à calculer exactement, on a donc recours à des heuristiques pour l'approcher.

C'est le **masque causal** du transformer qui rend cette décomposition possible : dans chaque couche d'attention, la position $i$ ne peut regarder que les positions $j \le i$. Pendant l'entraînement, le modèle apprend ainsi à prédire chaque token à partir des seuls tokens précédents : exactement ce dont la génération a besoin.

## Tokenisation

Le tokeniseur récupéré avec le modèle transforme une phrase en indices de vocabulaire (des *sous-mots*) que le modèle comprend. Dans l'affichage des tokens, `Ġ` marque un token qui commence par une espace.

Nous utiliserons dans la suite de la démo les fonctions `encode` et `decode` suivantes. `encode` ajoute le symbole de début de texte, et renvoie un tenseur de forme `(1, nombre de tokens)`.

BelGPT-2 a été entraîné sur des textes où la ponctuation est séparée des mots par des espaces (« j' ai », « fin . ») : ses générations en gardent la trace. Il colle aussi parfois deux phrases sans espace entre elles (« soir.Les ») : un défaut du modèle (des fins de ligne retirées de son corpus, sans doute), pas de la bibliothèque.

In [6]:
def encode(phrase: str):
  """Indices des tokens, après le symbole de début : forme (1, nombre de tokens)."""
  ids_tokens = keras.ops.convert_to_tensor(tokenizer(phrase), dtype="int32")
  debut = keras.ops.convert_to_tensor([tokenizer.start_token_id], dtype="int32")
  return keras.ops.expand_dims(keras.ops.concatenate([debut, ids_tokens]), axis=0)


def decode(ids_tokens) -> str:
  """Texte des tokens, sans le symbole de début."""
  return tokenizer.detokenize(ids_tokens[0, 1:])


exemple = encode(
  "Je pense, donc je suis" if langue == "fr" else "I think, therefore I am"
)
print(exemple)
print([tokenizer.id_to_token(int(i)) for i in keras.ops.convert_to_numpy(exemple[0])])

tf.Tensor([[   2 4274 2068   14  864  475 1057]], shape=(1, 7), dtype=int32)
['<|endoftext|>', 'Je', 'Ġpense', ',', 'Ġdonc', 'Ġje', 'Ġsuis']


## Étude des sorties du modèle pré-entraîné

La première étape pour décoder des phrases depuis un modèle appris est de faire une passe forward du modèle pour récupérer ses prédictions pour le prochain token. Étudions son comportement.

Comme tout modèle Keras, `causal_lm` s'appelle comme une fonction. Il prend un dictionnaire de deux tenseurs de forme `(batch, nombre de tokens)` :

- `"token_ids"` : les indices des tokens ;
- `"padding_mask"` : 1 pour un vrai token, 0 pour une position de remplissage (ici, nous n'avons que des 1, que donne [`keras.ops.ones_like`](https://keras.io/api/ops/numpy/#oneslike-function)).

Il renvoie les **logits** : les scores du modèle, avant softmax. Regardons leur forme pour le texte vide et pour « Je pense, donc je suis ».

In [7]:
# Le texte vide n'a qu'une position : Keras avertirait que la softmax de
# l'attention porte sur un axe de taille 1, ce qui est attendu ici
warnings.filterwarnings("ignore", message="You are using a softmax over axis")


def decrit_formes(phrase: str) -> None:
  ids_tokens = encode(phrase)
  logits = causal_lm(
    {"token_ids": ids_tokens, "padding_mask": keras.ops.ones_like(ids_tokens)}
  )
  print(f"Forme des logits pour '{phrase}' : {tuple(logits.shape)}")


decrit_formes("")
decrit_formes("Je pense, donc je suis" if langue == "fr" else "I think, therefore I am")

Forme des logits pour '' : (1, 1, 50257)
Forme des logits pour 'Je pense, donc je suis' : (1, 7, 50257)


Les logits ont la forme `(batch, nombre de tokens, taille du vocabulaire)` : pour chaque position de l'entrée, le modèle donne un score à chacun des 50 257 tokens du vocabulaire pour la position **suivante**. Le texte vide a une position, celle du symbole de début qu'`encode` ajoute. Pour générer, seule la dernière position nous intéresse.

## Fonction `propage` adaptée aux besoins du décodage

Nous utiliserons la fonction `propage` suivante, qui rend un résultat adapté au décodage : seule la dernière position nous intéresse (la prédiction du token suivant).

Elle utilise le cache de KerasHub (la méthode `call_with_cache` du modèle) : il garde en mémoire les clés et les valeurs d'attention des tokens déjà vus, ce qui évite de les recalculer à chaque nouveau token. Quand le cache est donné, il ne faut passer au modèle **que les nouveaux tokens**. Le cache a une taille fixe, `TAILLE_CACHE` positions, et `tf.function` compile le pas du modèle en graphe TensorFlow : le premier appel prend quelques secondes, les suivants sont rapides.

In [8]:
TAILLE_CACHE = 128  # positions que le cache peut garder : prompt et tokens générés


@tf.function
def appel_avec_cache(ids_tokens, cache, index):
  """Un pas du modèle sur un token, avec le cache, compilé par TensorFlow."""
  logits, _, cache = causal_lm.call_with_cache(ids_tokens, cache, index)
  return logits[:, -1, :], cache


def propage(ids_tokens, etat_cache=None):
  """Logits du token suivant, de forme (1, taille du vocabulaire), et le cache."""
  if etat_cache is None:
    backbone = causal_lm.backbone
    dim_tete = backbone.hidden_dim // backbone.num_heads
    forme = (1, backbone.num_layers, 2, TAILLE_CACHE, backbone.num_heads, dim_tete)
    etat_cache = (keras.ops.zeros(forme), 0)
  cache, vus = etat_cache
  # Un token à la fois : la fonction compilée voit toujours les mêmes formes
  for position in range(ids_tokens.shape[1]):
    logits, cache = appel_avec_cache(
      ids_tokens[:, position : position + 1], cache, tf.constant(vus + position)
    )
  return logits, (cache, vus + ids_tokens.shape[1])

Il faut donc utiliser au choix :

- `forward(all_tokens)`
- `forward(last_token, past)`

où `all_tokens` serait par exemple une séquence de 7 tokens de forme `(1, 7)` là où `last_token` serait de forme `(1, 1)`. Les logits renvoyés ont la forme `(1, taille du vocabulaire)` : la dimension du batch, de taille 1, est conservée.

Le modèle calcule pourtant des logits pour chaque position : ils servent à l'entraînement, où chaque position prédit le token qui la suit, toutes à la fois en une seule passe. Le masque causal empêche la position $i$ de voir les tokens suivants, sans quoi elle lirait directement le token qu'elle doit prédire.

## Boucle de décodage

Une boucle de décodage suit toujours le même schéma :

1. Passe forward sur ce qui a été produit jusqu'ici (entrées et sorties précédentes ; grâce au cache, seulement le dernier token après le premier pas)
2. Choix de l'indice du token suivant
3. Répétition de 1. et 2. jusqu'à ce qu'un critère d'arrêt soit satisfait (dans cette démo, seulement le nombre de tokens produits)

L'étape 2 est celle où tout se joue. Le code suivant implémente tout le reste, afin de pouvoir nous concentrer sur l'étape 2 : chaque méthode de décodage sera une fonction qui reçoit les logits du dernier token (forme `(1, taille du vocabulaire)`) et renvoie l'indice du token choisi (un tenseur de forme `()`). Décorée par `@decoding_loop`, elle devient une fonction `(prompt, nombre de tokens à générer, paramètres de la méthode)` qui renvoie le texte généré.

In [9]:
def boucle_decodage(fonction_pas):

  @functools.wraps(fonction_pas)
  def enveloppe(prompt: str, longueur: int, *args_pas, **kwargs_pas) -> str:
    decodes = [encode(prompt)]
    etat_cache = None
    for _ in tqdm.auto.trange(longueur, desc="Tokens", leave=False):
      logits, etat_cache = propage(decodes[-1], etat_cache)
      index = fonction_pas(logits, *args_pas, **kwargs_pas)
      decodes.append(keras.ops.cast(keras.ops.reshape(index, (1, 1)), "int32"))
    return decode(keras.ops.concatenate(decodes, axis=1))

  return enveloppe

## Décodage glouton (*greedy*)

Le décodage glouton est la méthode la plus simple pour décoder du texte depuis un modèle appris : à chaque étape, on prend le token de plus forte probabilité.

Dans l'équation :

$$P(\text{Texte}|\text{Initial}) = \prod_{i=1}^{\text{Taille du texte}} P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$$

le décodage glouton choisit $\text{Token}_i$ pour maximiser $P(\text{Token}_i| \text{Token}_{j<i}, \text{Initial})$. Or, il faut souvent choisir un token moins probable pour ensuite atteindre des tokens très probables.

L'exemple suivant est tiré de l'article de Hugging Face sur la génération de texte :

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

Après « The », on choisit « nice » parce que c'est le mot le plus probable, alors que le texte complet « The nice woman » a une probabilité (0,20) inférieure à celle qu'on aurait obtenue en choisissant « dog » (« The dog has », 0,36).

La fonction `glouton` renvoie l'indice du plus grand logit, avec [`keras.ops.argmax`](https://keras.io/api/ops/numpy/#argmax-function) : inutile de passer par la softmax, qui ne change pas l'ordre des scores. Générons 10 tokens après « Je suis allé au ».

In [10]:
@boucle_decodage
def glouton(logits):
  return keras.ops.argmax(logits, axis=-1)[0]


print(glouton("Je suis allé au" if langue == "fr" else "I went to the", 10))

Je suis allé au cinéma , j' ai vu des films , j


## Test sur des prompts variés

La fonction `teste_decodage` suivante applique une méthode de décodage, avec ses paramètres, à chacun des prompts de la liste `prompts`. Elle affiche les générations et les renvoie dans une liste ; `affiche` les affiche. Testons le décodage glouton avec 50 tokens par prompt.

> **Note au formateur** : GPT-2 et BelGPT-2 ont appris sur du texte du web non filtré. L'échantillonnage peut produire en direct un texte violent, politique ou choquant, même à partir d'amorces neutres comme celles-ci. Exécutez la démo avant la séance et relisez les générations, ou commentez les sorties stockées plutôt que de relancer les cellules devant le groupe.

In [11]:
if langue == "fr":
  prompts = [
    "Je suis allé au",
    "Le train pour Lyon",
    "Comment vas-tu",
    "La recette de la tarte aux pommes commence par",
    "Ça va merci, tu devrais",
    "Le chat de la voisine",
    "En 2050, les voitures",
  ]
else:
  prompts = [
    "I went to the",
    "The train to London",
    "How do you",
    "The apple pie recipe starts with",
    "I'm fine thank you, you should",
    "The neighbour's cat",
    "In 2050, cars",
  ]

In [12]:
def affiche(generations: list[str]) -> None:
  """Affiche chaque prompt de `inputs` avec sa génération."""
  for prompt, generation in zip(prompts, generations):
    print("—" * 80)
    print(f"Prompt : {prompt}")
    print(f"Génération : {generation}")
  print("—" * 80)


def teste_decodage(fonction, *args, **kwargs) -> list[str]:
  """Applique la méthode de décodage `function` à chaque prompt de `inputs`."""
  generations = [
    fonction(prompt, *args, **kwargs)
    for prompt in tqdm.auto.tqdm(prompts, desc="Prompts", leave=False)
  ]
  affiche(generations)
  return generations

In [13]:
generations_gloutonnes = teste_decodage(glouton, 50)

————————————————————————————————————————————————————————————————————————————————
Prompt : Je suis allé au
Génération : Je suis allé au cinéma , j' ai vu des films , j' ai vu des films , j' ai vu des films , j' ai vu des films , j' ai vu des films , j' ai vu des films , j' ai vu des films
————————————————————————————————————————————————————————————————————————————————
Prompt : Le train pour Lyon
Génération : Le train pour Lyon-Saint-Exupéry est arrivé à destination à 11h30 . Le premier est un peu plus long , mais il est très agréable à lire . Il est donc important de bien choisir son matériel . Il est donc important de bien choisir son matériel de
————————————————————————————————————————————————————————————————————————————————
Prompt : Comment vas-tu
Génération : Comment vas-tu ? Il est donc important de bien choisir son matériel de ski . Il est donc important de bien choisir son matériel de ski . Il est donc important de bien choisir son matériel de ski . Il est donc important de bien 

## Mesurer la répétition

Les générations gloutonnes « bouclent » vite : elles répètent les mêmes groupes de mots. Pour comparer les méthodes de décodage sur ce point, mesurons-le par la **part de 3-grammes distincts** d'un texte : on découpe le texte en mots, on forme tous les triplets de mots consécutifs (les 3-grammes), et on divise le nombre de triplets différents par le nombre total de triplets. Elle vaut 1 pour un texte sans répétition, et baisse quand des passages se répètent.

La fonction `mesure_repetition` suivante la calcule : `"a b c a b c"` a 4 triplets, dont 3 différents, soit 0,75. Les sections suivantes la réutilisent pour chaque méthode.

In [14]:
def mesure_repetition(texte: str) -> float:
  mots = texte.split()
  trigrammes = list(zip(mots, mots[1:], mots[2:]))
  return len(set(trigrammes)) / len(trigrammes)


print(mesure_repetition("a b c a b c"))
print(
  f"Glouton : {statistics.mean(map(mesure_repetition, generations_gloutonnes)):.2f}"
)

0.75
Glouton : 0.70


Plusieurs générations gloutonnes répètent la même phrase, ou le même groupe de mots, jusqu'à la fin : c'est une faiblesse très classique du décodage glouton. La part de 3-grammes distincts le chiffre : elle est nettement sous 1, là où un texte écrit par un humain en est proche.

## Mesurer la cohérence

La part de 3-grammes distincts mesure la répétition, pas la cohérence : un texte qui n'a plus de sens ne se répète pas non plus. Pour la cohérence, demandons au modèle lui-même. La **log-probabilité moyenne** d'un texte est la moyenne, sur ses tokens, de $\log P(\text{Token}_i | \text{Token}_{j<i})$ selon le modèle : plus elle est proche de 0, plus le modèle trouve le texte probable, et elle baisse quand le texte prend des tours improbables.

Elle se calcule en une seule passe forward sur tout le texte : les logits de la position $i$ donnent la distribution du token $i + 1$, dans laquelle on lit la log-probabilité du token réellement présent.

La fonction `calcule_log_prob` suivante la calcule, avec [`keras.ops.log_softmax`](https://keras.io/api/ops/nn/#logsoftmax-function) puis [`keras.ops.take_along_axis`](https://keras.io/api/ops/numpy/#takealongaxis-function), qui lit à chaque position la log-probabilité du token suivant réel. La fonction `mesure` affiche les deux mesures en moyenne sur les générations d'une méthode ; les sections suivantes l'utilisent pour chaque méthode.

In [15]:
def calcule_log_prob(texte: str) -> float:
  ids_tokens = encode(texte)
  logits = causal_lm(
    {"token_ids": ids_tokens, "padding_mask": keras.ops.ones_like(ids_tokens)}
  )
  # La position i prédit le token i + 1 : décalage d'une position
  log_probas = keras.ops.log_softmax(logits[0, :-1], axis=-1)
  ids_suivants = ids_tokens[0, 1:]
  choisies = keras.ops.take_along_axis(log_probas, ids_suivants[:, None], axis=-1)
  return float(keras.ops.mean(choisies))


def mesure(nom: str, generations: list[str]) -> None:
  """Affiche les deux mesures, en moyenne sur les générations d'une méthode."""
  repetition = statistics.mean(map(mesure_repetition, generations))
  log_proba = statistics.mean(map(calcule_log_prob, generations))
  print(
    f"{nom} : {repetition:.2f} de 3-grammes distincts, "
    f"log-probabilité moyenne {log_proba:.2f}"
  )


mesure("Glouton", generations_gloutonnes)

Glouton : 0.70 de 3-grammes distincts, log-probabilité moyenne -2.05


Le glouton a une log-probabilité moyenne élevée : il prend à chaque pas le token le plus probable, et une boucle, une fois lancée, devient de plus en plus probable pour le modèle. Une log-probabilité élevée ne suffit donc pas à faire un bon texte : les sections suivantes regardent les deux mesures ensemble.

## Échantillonnage (*sampling*)

Les résultats du décodage glouton bouclent rapidement et sont souvent très génériques. Les méthodes suivantes cherchent à corriger ces défauts.

La première consiste à **échantillonner** à partir des probabilités du modèle, plutôt que de toujours choisir le token le plus probable : un token de probabilité 0,3 est tiré 3 fois sur 10.

La fonction `echantillonnage` reprend la forme de `glouton`, mais tire le token suivant avec [`keras.random.categorical`](https://keras.io/api/random/random_ops/#categorical-function), qui applique elle-même la softmax aux logits. Comparons ses deux mesures à celles du décodage glouton.

In [16]:
@boucle_decodage
def echantillonnage(logits):
  return keras.random.categorical(logits, 1)[0, 0]


generations = teste_decodage(echantillonnage, 50)
mesure("Échantillonnage", generations)

————————————————————————————————————————————————————————————————————————————————
Prompt : Je suis allé au
Génération : Je suis allé au collègeIl t' est difficile d' être heureuseLes autres . Les examens ne le permirent pas d' en mettre sur les joues des premières victimes de l' accident . Parfois j' ignore si je peux séduire dès le premier regard ou bien quand
————————————————————————————————————————————————————————————————————————————————
Prompt : Le train pour Lyon
Génération : Le train pour Lyon : une étape à souligner S' arrêter à Dijon Monique RIBELOIT-FERLIN En tout état de cause , je serai dans les deux ! Le terme format PDF est un condensé de termes similaires vers des technologies analogues : Wasate le permet
————————————————————————————————————————————————————————————————————————————————
Prompt : Comment vas-tu
Génération : Comment vas-tu ta blague ? Ils ont réalisé qu' il y avait une fracture à l' entrée de la ville . Le tout accompagné de berlingots de moutarde et d' une mo

Les générations ne bouclent plus : la part de 3-grammes distincts est proche de 1. En contrepartie, elles perdent souvent le fil, parce qu'un token peu probable est tiré de temps en temps et que la suite du texte doit s'en accommoder : leur log-probabilité moyenne est nettement plus basse que celle du glouton.

## Température

Avant la softmax, on peut diviser les logits par une **température** $T$ :

- $T < 1$ accentue les écarts : le modèle choisit plus souvent les tokens les plus probables (plus cohérent, moins varié) ;
- $T > 1$ les aplatit : les tokens rares sont plus souvent tirés (plus varié, moins cohérent) ;
- $T \to 0$ revient au décodage glouton.

La fonction `echantillonnage_temperature` divise les logits par la température avant le tirage. Comparons les générations et leurs deux mesures pour $T = 0.5$ et $T = 1.5$.

In [17]:
@boucle_decodage
def echantillonnage_temperature(logits, temperature: float):
  return keras.random.categorical(logits / temperature, 1)[0, 0]


for temperature in [0.5, 1.5]:
  generations = teste_decodage(echantillonnage_temperature, 50, temperature=temperature)
  mesure(f"T = {temperature}", generations)

————————————————————————————————————————————————————————————————————————————————
Prompt : Je suis allé au
Génération : Je suis allé au cinéma , j' ai vu des films , j' ai vu des films , j' ai vu des films , j' ai vu des films , j' ai vu des films , j' ai vu des films . La nouvelle version de la plate
————————————————————————————————————————————————————————————————————————————————
Prompt : Le train pour Lyon
Génération : Le train pour Lyon n' est pas dans les cartons . Il y a un an , on a eu l' idée de mettre en place un groupe de travail . Le grand prix de la critique internationale a récompensé le film " Chouchou " , d' après son
————————————————————————————————————————————————————————————————————————————————
Prompt : Comment vas-tu
Génération : Comment vas-tu ? Il est important que vous vous sentiez à l' aise avec les autres chiens . Il est vrai qu' il y a eu beaucoup de morts , mais de nombreux blessés . Il se peut qu' il contienne quelques imprécisions par rapport à l'
————————————

À $T = 0.5$, les générations sont plus cohérentes : leur log-probabilité moyenne se rapproche de celle du glouton ; plus $T$ baisse, plus elles se rapprochent du décodage glouton et de ses boucles. À $T = 1.5$, elles n'ont presque plus de sens : des tokens rares sont tirés à chaque pas, et leur log-probabilité moyenne s'effondre. La part de 3-grammes distincts, elle, sépare beaucoup moins ces deux réglages : elle mesure la répétition, pas la cohérence. La température règle ce compromis entre cohérence et variété, que les deux mesures chiffrent.

## Top-k sampling

Une variation de l'échantillonnage consiste à ne tirer que parmi les `k` tokens les plus probables, pour éviter de générer un token très improbable.

La fonction `echantillonnage_k` garde les `k` plus grands logits avec [`keras.ops.top_k`](https://keras.io/api/ops/core/#topk-function), qui donne aussi leurs indices dans le vocabulaire. Elle tire une position parmi ces `k` (`[None, :]` rajoute la dimension du batch), puis renvoie l'indice du token correspondant. Testons-la pour `k = 20`.

In [18]:
@boucle_decodage
def echantillonnage_k(logits, k: int):
  valeurs, indices = keras.ops.top_k(logits[0], k=k)
  tire = keras.random.categorical(valeurs[None, :], 1)[0, 0]
  return indices[tire]


generations = teste_decodage(echantillonnage_k, 50, k=20)
mesure("Top-k", generations)

————————————————————————————————————————————————————————————————————————————————
Prompt : Je suis allé au
Génération : Je suis allé au bureau de poste pour le remboursement du produit . L' homme était dans son appartement lorsqu' il a vu cet homme . Une situation qui n' est pas sans rappeler les tensions entre Londres et Varsovie , qui ont mis au défi le Parlement de Strasbourg de
————————————————————————————————————————————————————————————————————————————————
Prompt : Le train pour Lyon
Génération : Le train pour Lyon passe par Lyon ... à moins d' un km , il n ' y a pas vraiment le choix . Le site est une mine d' informations pour les parents et les élèves de CE1 et CE2.Les informations sont disponibles sur : Les inscriptions
————————————————————————————————————————————————————————————————————————————————
Prompt : Comment vas-tu
Génération : Comment vas-tu ? Le premier est constitué d' un ensemble de cinq morceaux qui forment un tout cohérent . Mais , au-delà des différences régionale

Les tokens hors des 20 plus probables ne sont plus jamais tirés : les générations restent variées, mais déraillent moins souvent que par échantillonnage simple, et leur log-probabilité moyenne remonte.

## Top-p sampling

Une autre amélioration consiste à ne garder que les tokens les plus probables dont la probabilité cumulée atteint `p` (le *noyau* de la distribution), et pas les suivants. Le nombre de tokens candidats s'adapte alors à la confiance du modèle : peu de candidats quand un token domine, beaucoup quand la distribution est plate.

La fonction `echantillonnage_k_p` l'applique après un top-k. Elle calcule les probabilités des `k` tokens gardés avec [`keras.ops.softmax`](https://keras.io/api/ops/nn/#softmax-function), puis leurs sommes cumulées avec [`keras.ops.cumsum`](https://keras.io/api/ops/numpy/#cumsum-function), et garde les tokens jusqu'à celui qui franchit `p`, inclus. Par exemple, pour les probabilités `[0.5, 0.3, 0.15, 0.05]` et `p = 0.85`, les sommes cumulées sont `[0.5, 0.8, 0.95, 1.0]` : 2 sont sous `p`, on garde 3 tokens. Testons-la pour `k = 20` et `p = 0.85`.

In [19]:
@boucle_decodage
def echantillonnage_k_p(logits, k: int, p: float):
  valeurs, indices = keras.ops.top_k(logits[0], k=k)
  probabilites = keras.ops.softmax(valeurs)
  # Nombre de tokens nécessaires pour que la probabilité cumulée atteigne p
  nb_gardes = int(keras.ops.sum(keras.ops.cumsum(probabilites) < p)) + 1
  tire = keras.random.categorical(valeurs[None, :nb_gardes], 1)[0, 0]
  return indices[tire]


generations = teste_decodage(echantillonnage_k_p, 50, k=20, p=0.85)
mesure("Top-k et top-p", generations)

————————————————————————————————————————————————————————————————————————————————
Prompt : Je suis allé au
Génération : Je suis allé au centre commercial ( à gauche ) où j' ai vu une énorme pancarte , " Stop au tabac " . C' est un outil très pratique pour les débutants , c' est à dire qu' il permet d' avoir plusieurs profils en même temps sans
————————————————————————————————————————————————————————————————————————————————
Prompt : Le train pour Lyon
Génération : Le train pour Lyon , une étape incontournable ! Mais le plus inquiétant est que les personnes qui le fréquentent ne sont pas toujours les plus proches de l' Eglise , qui ne sont pas forcément les mieux disposées à la prière . Et pourtant , dans cette nouvelle vidéo , vous découvrirez
————————————————————————————————————————————————————————————————————————————————
Prompt : Comment vas-tu
Génération : Comment vas-tu réagir aux photos que tu as postées sur le mur ? Il s' est rendu dans plusieurs pays pour rencontrer des dirigeants

Quand le modèle est sûr de lui, le noyau ne garde qu'un ou deux tokens, et le texte suit la suite la plus naturelle ; quand il hésite, le tirage reste ouvert : la part de 3-grammes distincts reste proche de 1, et la log-probabilité moyenne proche de celle du top-k. C'est le réglage le plus courant en pratique, avec la température.

## Génération avec `generate`

En pratique, on n'écrit pas la boucle de décodage : la méthode [`generate`](https://keras.io/keras_hub/api/models/gpt2/gpt2_causal_lm/#generate-method) des modèles de KerasHub implémente toutes ces stratégies. Comme notre `propage`, elle garde en cache les clés et les valeurs d'attention ; elle compile en plus toute la boucle avec XLA : la première génération prend une trentaine de secondes de compilation, les suivantes sont rapides. Le choix du token suivant se règle par `causal_lm.compile(sampler=...)`, avec les échantillonneurs de [`keras_hub.samplers`](https://keras.io/keras_hub/api/samplers/) : `GreedySampler`, `RandomSampler`, `TopKSampler`, `TopPSampler`, qui ont tous un paramètre `temperature`.

Générons une centaine de tokens pour chaque prompt de `prompts`, en un seul appel de `generate` sur la liste (`max_length=110` compte les tokens du prompt) : d'abord avec un top-p sampling restreint aux 30 tokens les plus probables, de noyau `p = 0.95`, puis avec le décodage glouton. Le paramètre `seed` fixe les tirages de l'échantillonneur : chaque exécution redonne les mêmes textes.

In [20]:
echantillonneurs = [
  keras_hub.samplers.TopPSampler(p=0.95, k=30, seed=2),
  keras_hub.samplers.GreedySampler(),
]
for echantillonneur in echantillonneurs:
  causal_lm.compile(sampler=echantillonneur)
  generations = causal_lm.generate(prompts, max_length=110)
  affiche(generations)
  mesure(type(echantillonneur).__name__, generations)

————————————————————————————————————————————————————————————————————————————————
Prompt : Je suis allé au
Génération : Je suis allé au centre Pokémon le soir.Les jeux ont été faits.C' est bien simple , je suis tombé dessus en le lisant ! On y trouve aussi des documents historiques à l' appui de l' explication de la structure du site . Ce que dit la loi sur les retraites aujourd' hui . Et ce ne sont pas les " plus de 45 ans " qui vont la contredire , en la décrivant . " Ces dispositions ne sont pas applicables aux personnes physiques , ni aux personnes morales . Nous n' y retrouvons pas de problème particulier ,
————————————————————————————————————————————————————————————————————————————————
Prompt : Le train pour Lyon
Génération : Le train pour Lyon à 3.200 EurosCarefour-Raffle , qui reliait Lyon à Paris dans la nuit de samedi à dimanche ( heure de l' est ) , n' est pas en service . Le Conseil National des Villes est un organisme démocratique constitué de citoyens qui ont pris le pouvo

Sur une centaine de tokens, l'écart se creuse : la plupart des générations gloutonnes finissent par boucler, et leur part de 3-grammes distincts chute, alors que le top-p sampling reste varié. Le glouton garde pourtant la log-probabilité moyenne la plus haute : ses boucles sont très probables pour le modèle. Aucune des deux mesures ne suffit seule. C'est pourquoi les modèles de langage sont presque toujours décodés par échantillonnage, avec une température, un top-k et un top-p réglés pour l'usage visé.

## Un texte plus long

Pour finir, laissons le modèle écrire un texte plus long à partir d'un début d'histoire, avec le même top-p sampling et jusqu'à 200 tokens. `generate` s'arrête plus tôt si le modèle produit le symbole de fin de texte.

In [21]:
histoire = (
  "Il était une fois un petit robot qui"
  if langue == "fr"
  else "Once upon a time, a little robot"
)
causal_lm.compile(sampler=keras_hub.samplers.TopPSampler(p=0.95, k=30, seed=2))
print(causal_lm.generate(histoire, max_length=200))

Il était une fois un petit robot qui voulait apprendre l' oikonométrie . Les " nouvelles " études du Centre de recherches en études stratégiques sur les conflits d' intérêts ( CRISES ) publiées en décembre dernier ont révélé que plus de la moitié des dirigeants de firmes d' audit ont eu tendance à accorder plus de crédits aux dirigeants des groupes financiers qu' à leurs salariés . Il y a deux ans , à la naissance , j' étais enceinte de mon deuxième enfant . L' occasion pour le chef de l' Etat , Alassane Ouattara , d' annoncer le renforcement de la présence française dans le pays et le retour d' une ambassade à Kédougou . C' est bien d' une partie de la gauche qui s' en fout , mais elle a perdu plus que jamais . La vie des femmes est très diverse , avec de longs mois de vie commune . Dans ce contexte , le Président de la République vient d' adresser à la Commission européenne des recommandations à la France dans le


## À retenir

- À chaque étape, le modèle ne produit pas un mot mais un score pour chaque token de son vocabulaire (50 257 pour GPT-2 comme pour BelGPT-2) ; un token est un mot ou un morceau de mot.
- Le texte dépend de la manière de choisir parmi ces scores, le **décodage** : le même modèle donne des textes très différents selon le réglage.
- Le décodage glouton (toujours le plus probable) est déterministe, générique, et boucle vite.
- L'échantillonnage tire au sort selon les probabilités : plus de variété, mais des tokens parfois incohérents. Le top-k et le top-p coupent les choix improbables, la **température** règle l'audace.
- Deux mesures chiffrent ce compromis : la part de 3-grammes distincts (la répétition) et la log-probabilité moyenne (la cohérence, vue par le modèle). Aucune ne suffit seule : le glouton a la meilleure log-probabilité, et boucle.
- En pratique, on appelle une méthode `generate` (celle de KerasHub ici, celle de `transformers` en PyTorch), qui garde un cache d'attention, avec ces réglages.